In [ ]:
from CLEAN.utils import *
import torch
import torch.nn as nn
dirs = "../local_snapshot/tttianhao-CLEAN-0cf2cac/"
id_ec_dict, ec_id_dict = get_ec_id_dict(dirs+"data/split10_re2.csv")

In [ ]:
class LayerNormNet(nn.Module):
    def __init__(self, hidden_dim, out_dim, device, dtype, drop_out=0.1):
        super(LayerNormNet, self).__init__()
        self.hidden_dim1 = hidden_dim
        self.out_dim = out_dim
        self.drop_out = drop_out
        self.device = device
        self.dtype = dtype

        self.fc1 = nn.Linear(1024, hidden_dim, dtype=dtype, device=device)
        self.ln1 = nn.LayerNorm(hidden_dim, dtype=dtype, device=device)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim,
                             dtype=dtype, device=device)
        self.ln2 = nn.LayerNorm(hidden_dim, dtype=dtype, device=device)
        self.fc3 = nn.Linear(hidden_dim, out_dim, dtype=dtype, device=device)
        self.dropout = nn.Dropout(p=drop_out)

    def forward(self, x):
        

        x = self.dropout(self.ln1(self.fc1(x)))
        x = torch.relu(x)
        x = self.dropout(self.ln2(self.fc2(x)))
        x = torch.relu(x)
        x = self.fc3(x)
        return x

In [ ]:

def load_esm(lookup):
    esm = format_esm(torch.load(dirs+'/data/esm_data/' + lookup + '.pt'))
    return esm.unsqueeze(0)


def esm_embedding(ec_id_dict, device, dtype):
    '''
    Loading esm embedding in the sequence of EC numbers
    prepare for calculating cluster center by EC
    '''
    esm_emb = []
    # for ec in tqdm(list(ec_id_dict.keys())):
    for ec in list(ec_id_dict.keys()):
        ids_for_query = list(ec_id_dict[ec])
        esm_to_cat = [load_esm(id) for id in ids_for_query]
        esm_emb = esm_emb + esm_to_cat
    return torch.cat(esm_emb).to(device=device, dtype=dtype)


def model_embedding_test(id_ec_test, model, device, dtype):
    '''
    Instead of loading esm embedding in the sequence of EC numbers
    the test embedding is loaded in the sequence of queries
    then inferenced with model to get model embedding
    '''
    ids_for_query = list(id_ec_test.keys())
    esm_to_cat = [load_esm(id) for id in ids_for_query]
    esm_emb = torch.cat(esm_to_cat).to(device=device, dtype=dtype)
    model_emb = model(esm_emb)
    return model_emb


def infer_maxsep(train_data, test_data, report_metrics = False, 
                 pretrained=True, model_name=None):
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda:0" if use_cuda else "cpu")
    dtype = torch.float32
    id_ec_train, ec_id_dict_train = get_ec_id_dict(dirs+'/data/' + train_data + '.csv')
    id_ec_test, _ = get_ec_id_dict(dirs+'/data/' + test_data + '.csv')
    # load checkpoints
    # NOTE: change this to LayerNormNet(512, 256, device, dtype) 
    # and rebuild with [python build.py install]
    # if inferencing on model trained with supconH loss
    model = LayerNormNet(512, 128, device, dtype)
    
    if pretrained:
        try:
            checkpoint = torch.load(dirs+'/data/pretrained/'+ train_data +'.pth')
        except FileNotFoundError as error:
            raise Exception('No pretrained weights for this training data')
    else:
        try:
            checkpoint = torch.load(dirs+'/data/model/'+ model_name +'.pth')
        except FileNotFoundError as error:
            raise Exception('No model found!')
            
    model.load_state_dict(checkpoint)
    model.eval()
    # load precomputed EC cluster center embeddings if possible
    if train_data == "split70":
        emb_train = torch.load(dirs+'/data/pretrained/70.pt')
    elif train_data == "split100":
        emb_train = torch.load(dirs+'/data/pretrained/100.pt')
    else:
        emb_train = model(esm_embedding(ec_id_dict_train, device, dtype))
        
    emb_test = model_embedding_test(id_ec_test, model, device, dtype)
    eval_dist = get_dist_map_test(emb_train, emb_test, ec_id_dict_train, id_ec_test, device, dtype)
    seed_everything()
    eval_df = pd.DataFrame.from_dict(eval_dist)
    ensure_dirs(dirs+"/results")
    out_filename = dirs+"results/" +  test_data
    write_max_sep_choices(eval_df, out_filename)
    if report_metrics:
        pred_label = get_pred_labels(out_filename, pred_type='_maxsep')
        pred_probs = get_pred_probs(out_filename, pred_type='_maxsep')
        true_label, all_label = get_true_labels(dirs+'/data/' + test_data)
        pre, rec, f1, roc, acc = get_eval_metrics(
            pred_label, pred_probs, true_label, all_label)
        print("############ EC calling results using maximum separation ############")
        print('-' * 75)
        print(f'>>> total samples: {len(true_label)} | total ec: {len(all_label)} \n'
            f'>>> precision: {pre:.3} | recall: {rec:.3}'
            f'| F1: {f1:.3} | AUC: {roc:.3} ')
        print('-' * 75)

In [ ]:
m=torch.load(dirs+'/data/model/'+ 'split10_triplet' +'.pth')

In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
dtype = torch.float32
model = LayerNormNet(512, 128, device, dtype)
model.load_state_dict(m)

In [ ]:
id_ec_test, _ = get_ec_id_dict(dirs+'/data/' + 'new2' + '.csv')

In [ ]:
emb_test = model_embedding_test(id_ec_test, model, device, dtype)

In [ ]:
emb_test.shape

In [ ]:
m['fc1.weight'].shape

In [ ]:
infer_maxsep("split10_re2", "new2", report_metrics=True,
pretrained=False, model_name="split10_triplet")

In [ ]:
file_test = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/New_embeddings.h5'
file_train = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/some_embeddings.h5'
fasta_path = '../local_snapshot/data/New3Di.fasta'
uniprot_ids = []
with open(fasta_path, 'r' ) as fasta_f:
    for line in fasta_f:
        # get uniprot ID from header and create new entry
        if line.startswith('>'):
            uniprot_id = line.replace('>', '').strip()
            # replace tokens that are mis-interpreted when loading h5
            uniprot_id = uniprot_id.replace("/","_").replace(".","_")
            uniprot_ids.append(uniprot_id)
len(uniprot_ids)

In [ ]:
def csv_to_csv(csv_name,fasta_name,uniprot_ids):
    csvfile = open(csv_name, 'r')
    csvreader = csv.reader(csvfile, delimiter='\t')
    outfile = open(fasta_name, 'w')
    status = False
    
    for i, rows in enumerate(csvreader):
        if i > 0 and rows[0] in uniprot_ids:
            
            outfile.write(rows[0]+'\t'+rows[1]+'\t'+rows[2]+'\n')
                
                    
                    
csv_to_csv(dirs+"data/new.csv", dirs+"data/new2.csv",uniprot_ids)            

In [ ]:
from CLEAN.model import LayerNormNet
from CLEAN.distance_map import get_dist_map_test
import pandas as pd
from CLEAN.evaluate import *
import h5py

def load_prost5(file,lookup):
    with h5py.File(file, 'r') as hf:
        dataset_name = lookup 
        dataset = hf[dataset_name]  

   
        data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)

def pro_embedding(ec_id_dict, device, dtype):
    '''
    Loading esm embedding in the sequence of EC numbers
    prepare for calculating cluster center by EC
    '''
    pro_emb = []
    # for ec in tqdm(list(ec_id_dict.keys())):
    for ec in list(ec_id_dict.keys()):
        ids_for_query = list(ec_id_dict[ec])
        pro_to_cat = [load_prost5(file_train,id) for id in ids_for_query]
        pro_emb = pro_emb + pro_to_cat
    return torch.cat(pro_emb).to(device=device, dtype=dtype)


def model_embedding_test(id_ec_test, model, device, dtype):
    '''
    Instead of loading esm embedding in the sequence of EC numbers
    the test embedding is loaded in the sequence of queries
    then inferenced with model to get model embedding
    '''
    ids_for_query = list(id_ec_test.keys())
    pro_to_cat = [load_prost5(file_test,id) for id in ids_for_query]
    pro_emb = torch.cat(pro_to_cat).to(device=device, dtype=dtype)
    model_emb = model(pro_emb)
    return model_emb


def infer_maxsep(train_data, test_data, report_metrics = False, 
                 pretrained=True, model_name=None):
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda:0" if use_cuda else "cpu")
    dtype = torch.float32
    id_ec_train, ec_id_dict_train = get_ec_id_dict(dirs+'/data/' + train_data + '.csv')
    id_ec_test, _ = get_ec_id_dict(dirs+'/data/' + test_data + '.csv')
    # load checkpoints
    # NOTE: change this to LayerNormNet(512, 256, device, dtype) 
    # and rebuild with [python build.py install]
    # if inferencing on model trained with supconH loss
    model = LayerNormNet(512, 128, device, dtype)
    
    if pretrained:
        try:
            checkpoint = torch.load(dirs+'/data/pretrained/'+ train_data +'.pth')
        except FileNotFoundError as error:
            raise Exception('No pretrained weights for this training data')
    else:
        try:
            checkpoint = torch.load(dirs+'/data/model/'+ model_name +'.pth')
        except FileNotFoundError as error:
            raise Exception('No model found!')
            
    model.load_state_dict(checkpoint)
    model.eval()
    # load precomputed EC cluster center embeddings if possible
    if train_data == "split70":
        emb_train = torch.load(dirs+'/data/pretrained/70.pt')
    elif train_data == "split100":
        emb_train = torch.load(dirs+'/data/pretrained/100.pt')
    else:
        emb_train = model(pro_embedding(ec_id_dict_train, device, dtype))
        
    emb_test = model_embedding_test(id_ec_test, model, device, dtype)
    eval_dist = get_dist_map_test(emb_train, emb_test, ec_id_dict_train, id_ec_test, device, dtype)
    seed_everything()
    eval_df = pd.DataFrame.from_dict(eval_dist)
    ensure_dirs(dirs+"/results")
    out_filename = dirs+"results/" +  test_data
    write_max_sep_choices(eval_df, out_filename)
    if report_metrics:
        pred_label = get_pred_labels(out_filename, pred_type='_maxsep')
        pred_probs = get_pred_probs(out_filename, pred_type='_maxsep')
        true_label, all_label = get_true_labels(dirs+'/data/' + test_data)
        pre, rec, f1, roc, acc = get_eval_metrics(
            pred_label, pred_probs, true_label, all_label)
        print("############ EC calling results using maximum separation ############")
        print('-' * 75)
        print(f'>>> total samples: {len(true_label)} | total ec: {len(all_label)} \n'
            f'>>> precision: {pre:.3} | recall: {rec:.3}'
            f'| F1: {f1:.3} | AUC: {roc:.3} ')
        print('-' * 75)


In [ ]:
infer_maxsep("split10_re2", "new2", report_metrics=True,
pretrained=False, model_name="split10pro_triplet")

In [ ]:
file_test = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/sub3Di/new_sub3Di.h5'
file_train = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/sub3Di/sub3Di.h5'
def load_prost5(file,lookup):
    with h5py.File(file, 'r') as hf:
        dataset_name = lookup 
        dataset = hf[dataset_name]  

   
        data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)

def pro_embedding(ec_id_dict, device, dtype):
    '''
    Loading esm embedding in the sequence of EC numbers
    prepare for calculating cluster center by EC
    '''
    pro_emb = []
    # for ec in tqdm(list(ec_id_dict.keys())):
    for ec in list(ec_id_dict.keys()):
        ids_for_query = list(ec_id_dict[ec])
        pro_to_cat = [load_prost5(file_train,id) for id in ids_for_query]
        pro_emb = pro_emb + pro_to_cat
    return torch.cat(pro_emb).to(device=device, dtype=dtype)


def model_embedding_test(id_ec_test, model, device, dtype):
    '''
    Instead of loading esm embedding in the sequence of EC numbers
    the test embedding is loaded in the sequence of queries
    then inferenced with model to get model embedding
    '''
    ids_for_query = list(id_ec_test.keys())
    pro_to_cat = [load_prost5(file_test,id) for id in ids_for_query]
    pro_emb = torch.cat(pro_to_cat).to(device=device, dtype=dtype)
    model_emb = model(pro_emb)
    return model_emb


def infer_maxsep(train_data, test_data, report_metrics = False, 
                 pretrained=True, model_name=None):
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda:0" if use_cuda else "cpu")
    dtype = torch.float32
    id_ec_train, ec_id_dict_train = get_ec_id_dict(dirs+'/data/' + train_data + '.csv')
    id_ec_test, _ = get_ec_id_dict(dirs+'/data/' + test_data + '.csv')
    # load checkpoints
    # NOTE: change this to LayerNormNet(512, 256, device, dtype) 
    # and rebuild with [python build.py install]
    # if inferencing on model trained with supconH loss
    model = LayerNormNet(512, 128, device, dtype)
    
    if pretrained:
        try:
            checkpoint = torch.load(dirs+'/data/pretrained/'+ train_data +'.pth')
        except FileNotFoundError as error:
            raise Exception('No pretrained weights for this training data')
    else:
        try:
            checkpoint = torch.load(dirs+'/data/model/'+ model_name +'.pth')
        except FileNotFoundError as error:
            raise Exception('No model found!')
            
    model.load_state_dict(checkpoint)
    model.eval()
    # load precomputed EC cluster center embeddings if possible
    if train_data == "split70":
        emb_train = torch.load(dirs+'/data/pretrained/70.pt')
    elif train_data == "split100":
        emb_train = torch.load(dirs+'/data/pretrained/100.pt')
    else:
        emb_train = model(pro_embedding(ec_id_dict_train, device, dtype))
        
    emb_test = model_embedding_test(id_ec_test, model, device, dtype)
    eval_dist = get_dist_map_test(emb_train, emb_test, ec_id_dict_train, id_ec_test, device, dtype)
    seed_everything()
    eval_df = pd.DataFrame.from_dict(eval_dist)
    ensure_dirs(dirs+"/results")
    out_filename = dirs+"results/" +  test_data
    write_max_sep_choices(eval_df, out_filename)
    if report_metrics:
        pred_label = get_pred_labels(out_filename, pred_type='_maxsep')
        pred_probs = get_pred_probs(out_filename, pred_type='_maxsep')
        true_label, all_label = get_true_labels(dirs+'/data/' + test_data)
        pre, rec, f1, roc, acc = get_eval_metrics(
            pred_label, pred_probs, true_label, all_label)
        print("############ EC calling results using maximum separation ############")
        print('-' * 75)
        print(f'>>> total samples: {len(true_label)} | total ec: {len(all_label)} \n'
            f'>>> precision: {pre:.3} | recall: {rec:.3}'
            f'| F1: {f1:.3} | AUC: {roc:.3} ')
        print('-' * 75)


In [ ]:
infer_maxsep("split10_re2", "new2", report_metrics=True,
pretrained=False, model_name="split10_sub3Di_triplet")

In [ ]:
file_test = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/new_seq.h5'
file_train = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/split10_seq.h5'

def load_prost5(file,lookup):
    with h5py.File(file, 'r') as hf:
        dataset_name = lookup 
        dataset = hf[dataset_name]  

   
        data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)

def pro_embedding(ec_id_dict, device, dtype):
    '''
    Loading esm embedding in the sequence of EC numbers
    prepare for calculating cluster center by EC
    '''
    pro_emb = []
    # for ec in tqdm(list(ec_id_dict.keys())):
    for ec in list(ec_id_dict.keys()):
        ids_for_query = list(ec_id_dict[ec])
        pro_to_cat = [load_prost5(file_train,id) for id in ids_for_query]
        pro_emb = pro_emb + pro_to_cat
    return torch.cat(pro_emb).to(device=device, dtype=dtype)


def model_embedding_test(id_ec_test, model, device, dtype):
    '''
    Instead of loading esm embedding in the sequence of EC numbers
    the test embedding is loaded in the sequence of queries
    then inferenced with model to get model embedding
    '''
    ids_for_query = list(id_ec_test.keys())
    pro_to_cat = [load_prost5(file_test,id) for id in ids_for_query]
    pro_emb = torch.cat(pro_to_cat).to(device=device, dtype=dtype)
    model_emb = model(pro_emb)
    return model_emb


def infer_maxsep(train_data, test_data, report_metrics = False, 
                 pretrained=True, model_name=None):
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda:0" if use_cuda else "cpu")
    dtype = torch.float32
    id_ec_train, ec_id_dict_train = get_ec_id_dict(dirs+'/data/' + train_data + '.csv')
    id_ec_test, _ = get_ec_id_dict(dirs+'/data/' + test_data + '.csv')
    # load checkpoints
    # NOTE: change this to LayerNormNet(512, 256, device, dtype) 
    # and rebuild with [python build.py install]
    # if inferencing on model trained with supconH loss
    model = LayerNormNet(512, 128, device, dtype)
    
    if pretrained:
        try:
            checkpoint = torch.load(dirs+'/data/pretrained/'+ train_data +'.pth')
        except FileNotFoundError as error:
            raise Exception('No pretrained weights for this training data')
    else:
        try:
            checkpoint = torch.load(dirs+'/data/model/'+ model_name +'.pth')
        except FileNotFoundError as error:
            raise Exception('No model found!')
            
    model.load_state_dict(checkpoint)
    model.eval()
    # load precomputed EC cluster center embeddings if possible
    if train_data == "split70":
        emb_train = torch.load(dirs+'/data/pretrained/70.pt')
    elif train_data == "split100":
        emb_train = torch.load(dirs+'/data/pretrained/100.pt')
    else:
        emb_train = model(pro_embedding(ec_id_dict_train, device, dtype))
        
    emb_test = model_embedding_test(id_ec_test, model, device, dtype)
    eval_dist = get_dist_map_test(emb_train, emb_test, ec_id_dict_train, id_ec_test, device, dtype)
    seed_everything()
    eval_df = pd.DataFrame.from_dict(eval_dist)
    ensure_dirs(dirs+"/results")
    out_filename = dirs+"results/" +  test_data
    write_max_sep_choices(eval_df, out_filename)
    if report_metrics:
        pred_label = get_pred_labels(out_filename, pred_type='_maxsep')
        pred_probs = get_pred_probs(out_filename, pred_type='_maxsep')
        true_label, all_label = get_true_labels(dirs+'/data/' + test_data)
        pre, rec, f1, roc, acc = get_eval_metrics(
            pred_label, pred_probs, true_label, all_label)
        print("############ EC calling results using maximum separation ############")
        print('-' * 75)
        print(f'>>> total samples: {len(true_label)} | total ec: {len(all_label)} \n'
            f'>>> precision: {pre:.3} | recall: {rec:.3}'
            f'| F1: {f1:.3} | AUC: {roc:.3} ')
        print('-' * 75)

In [ ]:
infer_maxsep("split10_re2", "new2", report_metrics=True,
pretrained=False, model_name="split10proSEQ_triplet")